# Confidence Intervals for Proportions

Not every survey measures something in centimeters. Ask 30 Statistopians whether they own a car, hear "yes" 24 times, and your estimate is a **sample proportion**: $\hat{p} = 24/30 = 0.8$. The confidence interval keeps its familiar shape — $\hat{p} \pm \text{margin of error}$ — and only the **standard error** changes: for proportions it is $\sqrt{\hat{p}(1-\hat{p})/n}$, giving $0.8 \pm 1.96\sqrt{0.8 \times 0.2/30} = 0.8 \pm 0.14$, i.e. the interval $[0.66, 0.94]$. Underneath, nothing is truly new: a proportion is just the sample *mean* of a Bernoulli variable. It is based on the ninth video of Lesson 7 (Confidence Intervals).

## 1. Introduction

The last lesson's intervals were for the mean of a numeric measurement. This notebook transfers the machinery to yes/no questions — the domain of polls, conversion rates, and market shares — where the answer is a proportion between 0 and 1. One formula changes; the logic doesn't.

After completing this notebook you will be able to:

- Compute a sample proportion $\hat{p} = x/n$ and recognize it as a sample mean of 0/1 outcomes.
- Write the proportion interval $\hat{p} \pm z_{1-\alpha/2}\sqrt{\hat{p}(1-\hat{p})/n}$ and explain where the new standard error comes from.
- Reproduce the video's example: 24 of 30 car owners → $[0.66, 0.94]$ at 95%.
- Plan sample sizes for a target margin — including the pollster's worst-case trick $p = 0.5$.
- Probe the recipe's fine print: when the plug-in approximation is trustworthy and when small counts break it.

## 2. Theory

### From means to proportions

For sample means the interval was $\bar{x} \pm z_{1-\alpha/2}\,\sigma/\sqrt{n}$. Now the question changes flavor: *what proportion of Statistopia's adults own a car?* Sample $n = 30$ people; $x = 24$ of them own one. The **sample proportion** is

$$
\hat{p} = \frac{x}{n} = \frac{24}{30} = 0.8
$$

It's probably not the population proportion exactly — but it's likely close, and we want a 95% confidence interval around it.

### The same shape, a new standard error

$$
\text{confidence interval} = \hat{p} \pm \text{margin of error}, \qquad
\text{margin of error} = z_{1-\alpha/2}\,\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}
$$

The *whole* difference sits in the standard error:

| | sample means | sample proportions |
|---|---|---|
| estimate | $\bar{x}$ | $\hat{p}$ |
| standard error | $\dfrac{\sigma}{\sqrt{n}}$ | $\sqrt{\dfrac{\hat{p}(1-\hat{p})}{n}}$ |
| interval | $\bar{x} \pm z\,\dfrac{\sigma}{\sqrt{n}}$ | $\hat{p} \pm z\sqrt{\dfrac{\hat{p}(1-\hat{p})}{n}}$ |

### Why that formula — the Bernoulli connection

Code each answer as a number: 1 for "owns a car," 0 for "doesn't." Then $\hat{p}$ is literally the **sample mean** of these 0/1 values, and each answer is a Bernoulli($p$) variable with standard deviation $\sigma = \sqrt{p(1-p)}$. Drop that $\sigma$ into the familiar $\sigma/\sqrt{n}$ and out comes $\sqrt{p(1-p)/n}$ — with one honest plug-in: since $p$ is exactly what we don't know, we estimate the standard error with $\hat{p}$ in its place. The central limit theorem (Lesson 5, and notebook 03's Bernoulli exercise) supplies the approximate normality that justifies the z critical value.

### The worked example

$n = 30$, $\hat{p} = 0.8$, 95% confidence, so $z = 1.96$:

$$
\text{margin} = 1.96\sqrt{\frac{0.8 \times 0.2}{30}} = 1.96 \times 0.0730 = 0.14
$$

$$
\text{interval} = 0.8 \pm 0.14 = [0.66,\; 0.94]
$$

We are 95% confident (in the recipe sense) that the population's car-ownership proportion lies between 66% and 94%.

## 3. Worked Examples

### Example 1: the computation, digit by digit

$\hat{p}(1-\hat{p}) = 0.8 \times 0.2 = 0.16$; divide by $n$: $0.16/30 = 0.00533$; square root: $0.0730$ — the standard error. Times $1.96$: margin $0.143$. Interval: $[0.657, 0.943]$, which the video rounds to $[0.66, 0.94]$.

### Example 2: the standard error peaks at fifty-fifty

$\hat{p}(1-\hat{p})$ is largest at $\hat{p} = 0.5$ (value $0.25$) and shrinks toward 0 at the extremes:

| $\hat{p}$ | 0.5 | 0.8 | 0.95 |
|---|---|---|---|
| SE at $n = 30$ | 0.091 | 0.073 | 0.040 |

A fifty-fifty question is the *noisiest* — useful pessimism for planning (Example 3).

### Example 3: planning a poll

The sample-size formula from two notebooks ago, with the proportion SE inside:

$$
n \;\ge\; \frac{z_{1-\alpha/2}^2\; p(1-p)}{\text{MOE}^2}
$$

Before polling you don't know $p$, so use the worst case $p = 0.5$: for a margin of $\pm 5$ points at 95%, $n \ge 1.96^2 \times 0.25/0.05^2 = 384.2 \to 385$ — the origin of the classic "poll of about 400 people, margin ±5%." A news poll advertising ±3 points needs $n \ge 1068$.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(41)

def z_score(p):
    """Normal critical value z_p (probability p to the left)."""
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

def proportion_ci(x, n, level=0.95):
    """Confidence interval for a proportion from x successes in n trials."""
    p_hat = x / n
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    margin = z_score(1 - (1 - level) / 2) * se
    return p_hat, margin, (p_hat - margin, p_hat + margin)

### The video's example, narrated

In [ ]:
p_hat, margin, (lo, hi) = proportion_ci(x=24, n=30)
print(f"sample proportion: p-hat = 24/30 = {p_hat:.0%}")
print(f"standard error: sqrt(0.8 * 0.2 / 30) = {np.sqrt(0.8 * 0.2 / 30):.4f}")
print(f"margin of error: 1.96 * {np.sqrt(0.8 * 0.2 / 30):.4f} = {margin:.2f}")
print(f"95% confidence interval: [{lo:.2f}, {hi:.2f}]")

### A proportion is a Bernoulli sample mean — checked

Simulate the sampling distribution of $\hat{p}$ at $p = 0.8$, $n = 30$: its standard deviation should match $\sqrt{p(1-p)/n}$, and its histogram should sit under the CLT's normal curve (a little lumpy, since only 31 values of $\hat{p}$ are even possible).

In [ ]:
P_TRUE, n = 0.8, 30
answers = rng.random((100_000, n)) < P_TRUE      # 100k surveys of 30 yes/no answers
p_hats = answers.mean(axis=1)                     # each survey's p-hat IS a mean

theory_sd = np.sqrt(P_TRUE * (1 - P_TRUE) / n)
print(f"standard deviation of p-hat: simulated {p_hats.std():.4f}, "
      f"theory sqrt(p(1-p)/n) = {theory_sd:.4f}")

plt.figure(figsize=(9.5, 4.4))
plt.hist(p_hats, bins=np.arange(0.5, 1.02, 1 / n) - 1 / (2 * n), density=True,
         color="tab:green", alpha=0.6, edgecolor="white",
         label=r"simulated $\hat{p}$ (n = 30)")
x_axis = np.linspace(0.5, 1.0, 400)
plt.plot(x_axis, np.exp(-0.5 * ((x_axis - P_TRUE) / theory_sd) ** 2)
         / (theory_sd * np.sqrt(2 * np.pi)), color="black", linewidth=2,
         label=r"CLT: $N(p,\; p(1-p)/n)$")
plt.xlabel(r"sample proportion $\hat{p}$")
plt.ylabel("density")
plt.title("The sampling distribution of a proportion: a Bernoulli mean obeying the CLT")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Does the recipe deliver 95%? Almost — mind the fine print

Coverage of the plug-in recipe at the video's own parameters, and at a larger sample. At $n = 30$ the approximation runs a couple of points optimistic; the exercises chase down exactly when that happens.

In [ ]:
for n_test in (30, 100, 1000):
    surveys = (rng.random((50_000, n_test)) < P_TRUE).sum(axis=1)
    p_hats_t = surveys / n_test
    margins = z_score(0.975) * np.sqrt(p_hats_t * (1 - p_hats_t) / n_test)
    coverage = (np.abs(p_hats_t - P_TRUE) <= margins).mean()
    print(f"n = {n_test:>4d}: coverage = {coverage:.1%}")
print("\nClose to 95% -- but not exact, and (surprisingly) not even monotone in n:")
print("because p-hat can only take n+1 discrete values, the true coverage")
print("oscillates as n grows. Exercise 5 maps out where the recipe wobbles most.")

## 5. Visualizations

### The interval on the only axis that matters: 0 to 1

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.9))
ax.plot([0, 1], [0, 0], color="lightgray", linewidth=3, zorder=0)
ax.plot([lo, hi], [0, 0], color="tab:orange", linewidth=8,
        solid_capstyle="round", zorder=1)
ax.plot([p_hat], [0], marker="o", color="tab:blue", markersize=10, zorder=2)
ax.annotate(r"$\hat{p} = 0.80$", xy=(p_hat, 0.05), ha="center",
            fontsize=11, color="tab:blue")
for x, lab in [(lo, f"{lo:.2f}"), (hi, f"{hi:.2f}")]:
    ax.annotate(lab, xy=(x, -0.07), ha="center", fontsize=10)
for x in (0, 1):
    ax.annotate(str(x), xy=(x, -0.07), ha="center", fontsize=9, color="gray")
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-0.14, 0.12)
ax.set_yticks([])
ax.set_xlabel("proportion of car owners")
ax.set_title("24 yes in 30 asks: the population proportion is in [0.66, 0.94], with 95% confidence")
plt.tight_layout()
plt.show()

### The noisiest question is the fifty-fifty one

The standard error as a function of $\hat{p}$ — maximal at 0.5, vanishing at the extremes — and the margins it produces at $n = 30$.

In [ ]:
p_axis = np.linspace(0.01, 0.99, 300)
se_axis = np.sqrt(p_axis * (1 - p_axis) / 30)

plt.figure(figsize=(9.5, 4.4))
plt.plot(p_axis, z_score(0.975) * se_axis, color="tab:purple", linewidth=2,
         label="95% margin of error at n = 30")
for p_mark in (0.5, 0.8, 0.95):
    m_mark = z_score(0.975) * np.sqrt(p_mark * (1 - p_mark) / 30)
    plt.plot(p_mark, m_mark, "ko", markersize=5)
    offset = (0, -16) if p_mark == 0.5 else (0, 10)
    plt.annotate(f"$\\hat{{p}}$ = {p_mark}: ±{m_mark:.3f}", xy=(p_mark, m_mark),
                 xytext=offset, textcoords="offset points", ha="center", fontsize=9)
plt.xlabel(r"sample proportion $\hat{p}$")
plt.ylabel("margin of error")
plt.title(r"$\sqrt{\hat{p}(1-\hat{p})/n}$ peaks at fifty-fifty -- plan polls for the worst case")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** In a sample of 50 Statistopians, 35 own a bicycle. Compute $\hat{p}$ and the 95% confidence interval.

**Exercise 2.** Explain in two sentences why the proportion interval is the mean interval in disguise: what are the 0/1 variables, what is their $\sigma$, and which quantity gets plugged in where?

### Intermediate

**Exercise 3.** Recompute the video's car-ownership interval at 90% and at 99%. Which of these claims survive at each level: "a majority own cars" ($p > 0.5$)? "at least 70% own cars"?

**Exercise 4.** *Poll planning.* (a) Confirm the classic numbers: $n = 385$ for ±5 points and $n = 1068$ for ±3 points at 95%, worst case. (b) If a pilot study suggests $p \approx 0.8$, how many people does ±3 points really take — and what fraction of the worst-case budget is saved?

### Advanced

**Exercise 5.** *The jagged truth.* Fix $n = 30$ and sweep the true $p$ from 0.05 to 0.95: simulate the recipe's actual coverage at each $p$ and plot it. Where does it fall furthest below 95%? Relate your findings to the common rule of thumb that both $n\hat{p} \ge 10$ and $n(1-\hat{p}) \ge 10$ should hold — does the video's own example (24 and 6) pass?

**Exercise 6.** *Reading the news.* A poll reports a candidate at 52% support, "margin of error ±3 points, 95% confidence." (a) Roughly how many people were polled? (b) Does the poll show the candidate ahead of 50%? (c) What margin would the same poll deliver for a minor candidate polling at 5% — and why do pollsters quote the worst-case margin anyway?

## 7. Solutions

In [ ]:
# Exercise 1: bicycles
p1, m1, (lo1, hi1) = proportion_ci(35, 50)
print(f"p-hat = 35/50 = {p1:.0%}, margin = {m1:.3f}")
print(f"95% interval: [{lo1:.3f}, {hi1:.3f}]")

In [ ]:
# Exercise 2: the disguise
print("Code each answer as 1 (yes) or 0 (no): p-hat is the ordinary sample mean")
print("of those numbers, and each answer is Bernoulli(p) with sigma = sqrt(p(1-p)).")
print("Plug that sigma into the means formula sigma/sqrt(n) to get sqrt(p(1-p)/n),")
print("and -- since p is unknown -- estimate it with p-hat: the plug-in standard error.")

In [ ]:
# Exercise 3: other levels, and which claims survive
for level in (0.90, 0.95, 0.99):
    _, m3, (lo3, hi3) = proportion_ci(24, 30, level)
    majority = "yes" if lo3 > 0.5 else "no"
    seventy = "yes" if lo3 > 0.7 else "no"
    print(f"{level:.0%}: [{lo3:.3f}, {hi3:.3f}]  majority? {majority}   >= 70%? {seventy}")
print("\n'A majority owns cars' survives even at 99%; 'at least 70%' survives at")
print("none of them -- the lower limit dips below 0.7 already at 90%.")

In [ ]:
# Exercise 4: poll planning
z95 = z_score(0.975)
for moe in (0.05, 0.03):
    n_worst = int(np.ceil(z95**2 * 0.25 / moe**2))
    print(f"worst case (p = 0.5), MOE = ±{moe:.0%}: n = {n_worst}")

n_08 = int(np.ceil(z95**2 * 0.8 * 0.2 / 0.03**2))
print(f"\nwith pilot p ~ 0.8, MOE = ±3%: n = {n_08} "
      f"({1 - n_08 / 1068:.0%} fewer than the worst-case 1068)")

In [ ]:
# Exercise 5: coverage across p at n = 30
p_grid = np.linspace(0.05, 0.95, 46)
coverages = []
n5 = 30
for p_true in p_grid:
    counts = (rng.random((20_000, n5)) < p_true).sum(axis=1)
    ph = counts / n5
    mg = z_score(0.975) * np.sqrt(ph * (1 - ph) / n5)
    coverages.append((np.abs(ph - p_true) <= mg).mean())

plt.figure(figsize=(9.5, 4.4))
plt.plot(p_grid, coverages, "o-", color="tab:blue", markersize=3, linewidth=1.5)
plt.axhline(0.95, color="black", linestyle=":", label="promised 95%")
ok = (n5 * p_grid >= 10) & (n5 * (1 - p_grid) >= 10)
plt.axvspan(p_grid[ok].min(), p_grid[ok].max(), color="tab:green", alpha=0.12,
            label=r"rule of thumb: $np \geq 10$ and $n(1-p) \geq 10$")
plt.xlabel("true proportion p")
plt.ylabel("actual coverage at n = 30")
plt.title("Exercise 5: the plug-in recipe wobbles, and craters for extreme p")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

print(f"video's example: n*p-hat = 24, n*(1-p-hat) = 6 -- the second FAILS the")
print(f"rule of thumb, matching the dip visible around p = 0.8-0.9.")

In [ ]:
# Exercise 6: reading the news
z95 = z_score(0.975)
n_poll = z95**2 * 0.25 / 0.03**2
print(f"(a) ±3 points worst case -> n = {n_poll:.0f} ~ 1,050-1,100 people.")

_, m52, (lo52, hi52) = proportion_ci(int(0.52 * 1068), 1068)
print(f"(b) 52% ± {m52:.1%} -> [{lo52:.1%}, {hi52:.1%}]: the interval straddles 50%,")
print("    so 'ahead' is within the margin of error -- not a statistical lead.")

m05 = z95 * np.sqrt(0.05 * 0.95 / 1068)
print(f"(c) at 5% support the same poll's margin is only ±{m05:.1%} -- but pollsters")
print("    quote the worst-case ±3 so one number honestly covers every question asked.")

## 8. Conclusion

- Yes/no questions estimate a **proportion** $\hat{p} = x/n$; the interval keeps its shape, $\hat{p} \pm z_{1-\alpha/2}\sqrt{\hat{p}(1-\hat{p})/n}$, with only the standard error changed.
- No new theory hides underneath: a proportion **is** the sample mean of 0/1 Bernoulli variables with $\sigma = \sqrt{p(1-p)}$, estimated by plugging in $\hat{p}$ — and the CLT supplies the normality behind the z-score.
- The video's numbers: 24 of 30 car owners → $0.8 \pm 1.96 \times 0.073 = [0.66, 0.94]$ at 95%.
- The SE peaks at $\hat{p} = 0.5$, which powers the pollster's worst-case planning: ±5 points needs 385 people, ±3 points needs 1,068 — and a reported "±3" is the fifty-fifty worst case, generous for lopsided questions.
- Fine print worth respecting: the plug-in recipe is approximate — trust it when $n\hat{p} \ge 10$ *and* $n(1-\hat{p}) \ge 10$ (the video's own example, with only 6 non-owners, sits right at the edge, and its true coverage runs a couple of points shy).